# Run Experiments: Feature Testing, Promotion, and the Dashboard

This notebook is the **operational entry point** for ta_lab2's feature experimentation framework.
It shows you how to:

1. Define new candidate features in `configs/experiments/features.yaml`
2. Load them into a `FeatureRegistry` and resolve computation order via the DAG
3. Run information-coefficient (IC) experiments against live market data
4. Interpret Benjamini-Hochberg (BH) corrected p-values and the promotion gate
5. Batch-run all experimental features and compare them in a heatmap
6. Query persisted results from `cmc_feature_experiments`
7. Launch the interactive Streamlit dashboard for ongoing monitoring

---

## Table of Contents

1. [Prerequisites](#prerequisites)
2. [Setup](#setup)
3. [Parameters](#parameters)
4. [DB Connection and Validation](#db-connection)
5. [Part 1 — Feature Registry](#part1-registry)
6. [Part 2 — Dependency DAG](#part2-dag)
7. [Part 3 — Running a Single Experiment](#part3-single)
8. [Part 4 — Batch Experiments](#part4-batch)
9. [Part 5 — Loading Persisted Results](#part5-persisted)
10. [Part 6 — Launch the Dashboard](#part6-dashboard)
11. [Summary and What's Next](#summary)

<a id='prerequisites'></a>
## Prerequisites

**Required database tables:**
- `cmc_features` — bar-level feature store (id, ts, tf PK; 112 columns)
- `cmc_price_bars_multi_tf_u` — OHLCV price bars across all timeframes
- `cmc_vol` — realized volatility estimates (vol_7d, vol_30d, etc.)
- `cmc_ta_daily` — TA indicators including rsi_14
- `cmc_feature_experiments` — persisted experiment results (IC + BH rows)
- `dim_feature_registry` — promoted/deprecated feature lifecycle tracking
- `asset_data_coverage` — data coverage summary per asset/TF

**Required config:**
- `configs/experiments/features.yaml` — YAML feature definitions with lifecycle labels

**Python environment:**
- `ta_lab2` installed or available via `sys.path` (handled below)
- `scipy`, `pandas`, `numpy`, `plotly`, `matplotlib` in the active environment

> **Note:** All experiment runs in this notebook use `dry_run=True`.  
> Feature values are computed and scored but **not written to production tables**.  
> To persist results, use the CLI: `python -m ta_lab2.scripts.run_experiment --feature <name>`

In [ ]:
import sys
from pathlib import Path

_ROOT = Path.cwd().parent
if str(_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(_ROOT / "src"))

import helpers
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import plotly.graph_objects as go

import ta_lab2
print(f"ta_lab2 loaded from: {ta_lab2.__file__}")
print(f"Python: {sys.version}")
print(f"Pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

<a id='parameters'></a>
## Parameters

Adjust these variables to test against a different asset, timeframe, or date window.  
Re-run all cells after changing parameters — the results will update automatically.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `ASSET_ID` | `1` | Database asset ID (1 = BTC) |
| `TF` | `"1D"` | Timeframe string (`"1D"`, `"1W"`, `"4H"`, ...) |
| `START_DATE` | `"2021-01-01"` | Inclusive start of evaluation window |
| `END_DATE` | `"2025-12-31"` | Inclusive end of evaluation window |
| `HORIZONS` | `[1, 5, 20]` | Forward-bar horizons for IC calculation |
| `DRY_RUN` | `True` | Skip writing to production tables |
| `REGISTRY_PATH` | (auto) | Path to `configs/experiments/features.yaml` |

In [ ]:
# --- Parameters (change these to explore different assets/timeframes) ---
ASSET_ID    = 1            # BTC; change to 2 for ETH, etc.
TF          = "1D"         # Daily timeframe
START_DATE  = "2021-01-01"
END_DATE    = "2025-12-31"
HORIZONS    = [1, 5, 20]   # 1-day, 5-day, 20-day forward IC
DRY_RUN     = True         # Do not write to production tables

REGISTRY_PATH = str(_ROOT / "configs" / "experiments" / "features.yaml")

# Timestamps for ExperimentRunner (must be tz-aware UTC)
TRAIN_START = pd.Timestamp(START_DATE, tz="UTC")
TRAIN_END   = pd.Timestamp(END_DATE,   tz="UTC")

print(f"Asset ID  : {ASSET_ID}")
print(f"Timeframe : {TF}")
print(f"Window    : {START_DATE} -> {END_DATE}")
print(f"Horizons  : {HORIZONS}")
print(f"Dry run   : {DRY_RUN}")
print(f"Registry  : {REGISTRY_PATH}")

<a id='db-connection'></a>
## DB Connection and Validation

We create a single `NullPool` engine (no connection pooling) and validate that  
sufficient data exists for the selected asset and timeframe before running experiments.

In [ ]:
engine = helpers.get_engine()

check = helpers.validate_asset_data(engine, ASSET_ID, TF, min_days=365)
if not check["valid"]:
    print(f"WARNING: {check['message']}")
    print("Some experiment cells may produce empty results.")
else:
    print(f"Data coverage OK: {check['message']}")

<a id='part1-registry'></a>
## Part 1 — Feature Registry

### What Is the Feature Registry?

The `FeatureRegistry` is a YAML-driven catalog of **candidate features**.  
Each entry describes:

- **Lifecycle** (`experimental`, `promoted`, `deprecated`) — controls whether ExperimentRunner scores it
- **Compute spec** — either an inline pandas expression or a dotpath function reference
- **Input sources** — which DB tables and columns to load
- **Parameter sweeps** — optional `params` dict that produces named variants automatically
- **Tags** — free-form labels for filtering and grouping

Loading the registry:
1. Validates all expression syntax via `ast.parse`
2. Validates dotpath functions via `importlib`
3. Expands parameter sweeps into individual named variants
4. Computes a SHA-256 digest per spec (detects YAML changes between runs)

The registry is **read-only** — promotion writes to `dim_feature_registry` in the DB.

In [ ]:
from ta_lab2.experiments import FeatureRegistry

registry = FeatureRegistry(REGISTRY_PATH)
registry.load()

all_features    = registry.list_all()
experimental    = registry.list_experimental()

print(f"Registry loaded: {REGISTRY_PATH}")
print(f"Total features (including sweep variants): {len(all_features)}")
print(f"Experimental features: {len(experimental)}")
print()
print("All feature names:")
for name in sorted(all_features):
    spec = all_features[name]
    lifecycle = spec.get("lifecycle", "experimental")
    tags = spec.get("tags", [])
    digest = spec.get("yaml_digest", "")[:8]
    print(f"  [{lifecycle:>12s}]  {name:<35s}  tags={tags}  digest={digest}")

### Reading the YAML Definition

Below is the raw YAML for the registry file.  
Each feature entry follows this structure:

```yaml
features:
  <feature_name>:
    lifecycle: experimental     # or 'promoted' / 'deprecated'
    description: "..."
    compute:
      mode: inline              # or 'dotpath'
      expression: "col_a / col_b"  # pandas eval expression
    inputs:
      - table: cmc_vol          # must be in _ALLOWED_TABLES
        columns: [col_a, col_b]
    params:                     # optional — triggers sweep expansion
      period: [5, 14, 21]
    tags: [vol, ratio]
```

Parameter sweeps produce variants named `<base>_<key><value>` — e.g.  
`ret_vol_ratio` with `period: [5, 14, 21]` becomes  
`ret_vol_ratio_period5`, `ret_vol_ratio_period14`, `ret_vol_ratio_period21`.

In [ ]:
# Display the raw YAML for reference
with open(REGISTRY_PATH, encoding="utf-8") as f:
    yaml_content = f.read()

print(yaml_content)

In [ ]:
# Inspect a single feature spec in detail
if experimental:
    sample_name = experimental[0]
    spec = registry.get_feature(sample_name)
    print(f"Feature spec: '{sample_name}'")
    print("-" * 50)
    for key, val in spec.items():
        if key == "name":
            continue
        print(f"  {key:<20s}: {val}")
else:
    print("No experimental features found in registry.")

<a id='part2-dag'></a>
## Part 2 — Dependency DAG

### Why a DAG?

Experimental features can depend on other features.  
For example, a "return-normalized-by-regime-vol" feature might require  
a "regime-vol" base feature to be computed first.

The `resolve_experiment_dag()` function uses Python 3.9's `graphlib.TopologicalSorter`  
to determine a valid execution order where all dependencies are satisfied before dependents.  

**Key properties:**
- Features with no dependencies appear first (or in any order among themselves)
- A `graphlib.CycleError` is raised for circular dependencies
- External dependencies (promoted features, DB tables) are silently ignored —  
  only intra-registry edges affect ordering

In [ ]:
from ta_lab2.experiments import resolve_experiment_dag

ordered_names = resolve_experiment_dag(all_features)

print(f"Topological execution order ({len(ordered_names)} features):")
print()
for i, name in enumerate(ordered_names, 1):
    spec = all_features[name]
    deps = spec.get("depends_on", [])
    dep_str = " <- depends on: " + ", ".join(deps) if deps else ""
    print(f"  {i:2d}. {name}{dep_str}")

In [ ]:
# Build a dependency table for display
dag_rows = []
for i, name in enumerate(ordered_names, 1):
    spec = all_features[name]
    deps = spec.get("depends_on", [])
    dag_rows.append({
        "order": i,
        "feature": name,
        "lifecycle": spec.get("lifecycle", "experimental"),
        "compute_mode": spec.get("compute", {}).get("mode", "—"),
        "depends_on": ", ".join(deps) if deps else "(none)",
        "tags": ", ".join(spec.get("tags", [])),
    })

dag_df = pd.DataFrame(dag_rows).set_index("order")

# Style by lifecycle
def _style_lifecycle(val):
    colors = {
        "experimental": "background-color: #fff3cd; color: #856404",
        "promoted":     "background-color: #d4edda; color: #155724",
        "deprecated":   "background-color: #f8d7da; color: #721c24",
    }
    return colors.get(val, "")

dag_df.style.applymap(_style_lifecycle, subset=["lifecycle"])

**Reading the DAG table:**

- **order** — The topological execution position. Lower numbers run first.
- **lifecycle** — Yellow = experimental (will be scored); Green = promoted; Red = deprecated.
- **compute_mode** — `inline` uses a pandas expression; `dotpath` calls a Python function.
- **depends_on** — Features that must be computed before this one.

In the current registry all features are independent (no intra-registry dependencies),  
so the ordering is arbitrary. Once you add a feature that depends on `vol_ratio_30_7`,  
the resolver will place it after `vol_ratio_30_7` automatically.

<a id='part3-single'></a>
## Part 3 — Running a Single Experiment

### How ExperimentRunner Works

For each feature, `ExperimentRunner.run()` executes this pipeline:

```
Load inputs from declared source tables
       ↓
Compute feature via inline eval or dotpath
       ↓
Write values to session-scoped TEMP table  (skipped when dry_run=True)
       ↓
Score with compute_ic()  (Phase 37 IC engine)
       ↓
Apply BH correction across ALL rows (all assets × horizons × return_types)
       ↓
Return DataFrame with IC + BH p-values + cost metadata
```

**Key design decisions:**
- No production tables are written to during experiments (only a session TEMP table)
- BH correction is applied ONCE across all hypotheses in a single run — not per-asset
- Cost tracking (wall clock, peak memory, row count) is always returned

In [ ]:
from ta_lab2.experiments import ExperimentRunner

runner = ExperimentRunner(registry, engine)

# Run the first experimental feature
if experimental:
    target_feature = experimental[0]
    print(f"Running experiment for: '{target_feature}'")
    print(f"  Asset IDs : [{ASSET_ID}]")
    print(f"  TF        : {TF}")
    print(f"  Window    : {TRAIN_START.date()} to {TRAIN_END.date()}")
    print(f"  Horizons  : {HORIZONS}")
    print(f"  Dry run   : {DRY_RUN}")
    print()

    try:
        result_df = runner.run(
            target_feature,
            asset_ids=[ASSET_ID],
            tf=TF,
            train_start=TRAIN_START,
            train_end=TRAIN_END,
            horizons=HORIZONS,
            dry_run=DRY_RUN,
        )

        if result_df.empty:
            print("No results returned. Check that the source tables have data for this asset/TF/window.")
        else:
            print(f"Results shape: {result_df.shape}")
            # Cost metadata
            cost_cols = ["wall_clock_seconds", "peak_memory_mb", "n_rows_computed"]
            for col in cost_cols:
                if col in result_df.columns:
                    print(f"  {col}: {result_df[col].iloc[0]}")
    except Exception as exc:
        print(f"Experiment failed: {exc}")
        result_df = pd.DataFrame()
else:
    print("No experimental features in registry.")
    result_df = pd.DataFrame()

### Understanding the Results

Each row in `result_df` represents one **hypothesis** being tested:

| Column | Description |
|--------|-------------|
| `horizon` | Forward bars (e.g. 1 = next-bar return, 5 = 5-bar return) |
| `return_type` | `arith` (arithmetic %) or `log` (log return) |
| `ic` | Spearman rank correlation between feature and forward return |
| `ic_t_stat` | t-statistic for the IC estimate |
| `ic_p_value` | Two-tailed p-value for IC ≠ 0 (uncorrected) |
| `ic_p_value_bh` | Benjamini-Hochberg adjusted p-value |
| `ic_ir` | IC Information Ratio (IC / IC_std) |
| `n_obs` | Number of observations used |
| `yaml_digest` | SHA-256 digest of the YAML spec (detects changes) |

**Rule of thumb for IC:**
- |IC| > 0.02 — weak predictive signal
- |IC| > 0.05 — moderate signal (viable for quant strategies)
- |IC| > 0.10 — strong signal (rare in crypto, investigate carefully)

In [ ]:
# Display styled results
if not result_df.empty:
    display_cols = [
        c for c in ["horizon", "return_type", "ic", "ic_t_stat",
                    "ic_p_value", "ic_p_value_bh", "ic_ir", "n_obs"]
        if c in result_df.columns
    ]
    display_df = result_df[display_cols].copy()

    def _highlight_bh(val):
        if pd.isna(val):
            return ""
        return "background-color: #d4edda; font-weight: bold" if val < 0.05 else ""

    styler = helpers.style_ic_table(display_df)
    if "ic_p_value_bh" in display_df.columns:
        styler = styler.applymap(_highlight_bh, subset=["ic_p_value_bh"])
        styler = styler.format({"ic_p_value_bh": "{:.4f}"})

    display(styler)
else:
    print("No results to display.")

### The Promotion Gate: Benjamini-Hochberg Correction

When you run one feature against multiple horizons and return types, you are testing  
**multiple hypotheses simultaneously**. If you use a naive p < 0.05 threshold,  
you expect 5% false positives by chance alone.

ta_lab2 uses the **Benjamini-Hochberg (BH) procedure** to control the  
**False Discovery Rate (FDR)** — the expected fraction of rejections that are false.

**How BH works:**
1. Sort all p-values in ascending order: p(1) ≤ p(2) ≤ … ≤ p(m)
2. Find the largest k such that p(k) ≤ (k/m) × α
3. Reject all hypotheses up to and including k

BH is applied **once across all rows** of a single experiment run  
(all assets × horizons × return types) — not per-asset.

**Promotion gate:**  
A feature passes the gate if **at least one** BH-adjusted p-value < α (default 0.05).  
You can require a higher fraction by setting `min_pass_rate` on `FeaturePromoter`.

In [ ]:
from ta_lab2.experiments import FeaturePromoter, PromotionRejectedError

BH_ALPHA = 0.05

if not result_df.empty and "ic_p_value" in result_df.columns:
    promoter = FeaturePromoter(engine, registry)
    passed, enriched_df, reason = promoter.check_bh_gate(result_df, alpha=BH_ALPHA)

    print(f"Feature: '{target_feature}'")
    print(f"BH gate result: {'PASSED' if passed else 'REJECTED'}")
    print(f"Reason: {reason}")

    if "ic_p_value_bh" in enriched_df.columns:
        n_sig = (enriched_df["ic_p_value_bh"].dropna() < BH_ALPHA).sum()
        n_total = enriched_df["ic_p_value_bh"].notna().sum()
        print(f"Significant combos: {n_sig}/{n_total} at alpha={BH_ALPHA}")
else:
    print("No result data available for BH gate check.")
    passed = False

<a id='part4-batch'></a>
## Part 4 — Batch Experiments

Run all experimental features in topological order and collect IC results.  
Each feature is wrapped in `try/except` so a failure in one does not stop the batch.

After the batch completes, we visualize IC across features and horizons  
using a heatmap (Pandas Styler with diverging color scale).

In [ ]:
batch_results: dict[str, pd.DataFrame] = {}
batch_errors:  dict[str, str] = {}

# Run in topological order — respects depends_on
experimental_ordered = [n for n in ordered_names if n in experimental]

print(f"Running {len(experimental_ordered)} experimental features in DAG order...")
print()

for feat_name in experimental_ordered:
    print(f"  [{feat_name}] ... ", end="", flush=True)
    try:
        df = runner.run(
            feat_name,
            asset_ids=[ASSET_ID],
            tf=TF,
            train_start=TRAIN_START,
            train_end=TRAIN_END,
            horizons=HORIZONS,
            dry_run=DRY_RUN,
        )
        if df.empty:
            print("empty (no data)")
        else:
            batch_results[feat_name] = df
            n_sig = 0
            if "ic_p_value_bh" in df.columns:
                n_sig = int((df["ic_p_value_bh"].dropna() < BH_ALPHA).sum())
            print(f"OK  ({len(df)} rows, {n_sig} BH-sig)")
    except Exception as exc:
        batch_errors[feat_name] = str(exc)
        print(f"ERROR: {exc}")

print()
print(f"Completed: {len(batch_results)}/{len(experimental_ordered)} features")
if batch_errors:
    print(f"Errors   : {len(batch_errors)} features failed")
    for k, v in batch_errors.items():
        print(f"  {k}: {v}")

In [ ]:
# Build IC summary heatmap: rows = features, cols = horizon
if batch_results:
    heatmap_rows = []
    for feat_name, df in batch_results.items():
        if df.empty or "ic" not in df.columns:
            continue
        for _, row in df.iterrows():
            heatmap_rows.append({
                "feature": feat_name,
                "horizon": row.get("horizon"),
                "return_type": row.get("return_type", "arith"),
                "ic": row.get("ic"),
                "ic_p_value_bh": row.get("ic_p_value_bh"),
            })

    if heatmap_rows:
        hm_df = pd.DataFrame(heatmap_rows)

        # Use arithmetic returns only for clarity
        hm_arith = hm_df[hm_df["return_type"] == "arith"].copy()

        # Pivot: features x horizons
        pivot = hm_arith.pivot_table(index="feature", columns="horizon", values="ic")
        pivot.columns = [f"h={c}" for c in pivot.columns]

        # Pandas Styler with diverging color scale
        styled = pivot.style.background_gradient(
            cmap="RdYlGn", vmin=-0.10, vmax=0.10
        ).format("{:.4f}").set_caption(
            f"IC Heatmap — {TF} | Asset {ASSET_ID} | {START_DATE} to {END_DATE} (arith returns)"
        )
        display(styled)
    else:
        print("No IC data for heatmap.")
else:
    print("No batch results available. Check that source tables have data for this asset/TF/window.")

**Reading the heatmap:**

- **Green cells** — positive IC (feature positively correlates with forward returns at that horizon)
- **Red cells** — negative IC (contrarian signal at that horizon)
- **Yellow/neutral** — near-zero IC (no predictive power detected)
- **Bold green** in BH column — statistically significant after multiple-testing correction

A feature worth promoting typically shows:
- Consistent IC sign across horizons (not random)
- At least one horizon with |IC| > 0.03 and BH-adjusted p < 0.05
- Stable IC across different assets (not just one outlier)

If all features show near-zero IC, check that the source tables have sufficient data  
for the selected asset/TF/window, and that the features are computing correctly.

<a id='part5-persisted'></a>
## Part 5 — Loading Persisted Results

The `ExperimentRunner` can persist results to `cmc_feature_experiments` via the CLI.  
This allows you to:
- Track IC evolution over time as new data arrives
- Compare the same feature across different assets/TFs
- Use `FeaturePromoter.promote_feature()` which reads from this table

The cells below query the DB for any previously persisted results.

In [ ]:
from sqlalchemy import text as _text

with engine.connect() as conn:
    try:
        exp_df = pd.read_sql(
            _text("""
                SELECT
                    feature_name,
                    asset_id,
                    tf,
                    horizon,
                    return_type,
                    ic,
                    ic_p_value,
                    n_obs,
                    run_ts
                FROM cmc_feature_experiments
                ORDER BY run_ts DESC, feature_name, horizon
                LIMIT 100
            """),
            conn,
        )
    except Exception as exc:
        print(f"Could not query cmc_feature_experiments: {exc}")
        exp_df = pd.DataFrame()

if exp_df.empty:
    print("No persisted experiment results found.")
    print("Run experiments via CLI to populate this table:")
    print("  python -m ta_lab2.scripts.run_experiment --feature vol_ratio_30_7 --asset 1")
else:
    print(f"Loaded {len(exp_df)} persisted experiment rows")
    print(f"Features: {sorted(exp_df['feature_name'].unique())}")
    display(helpers.style_ic_table(exp_df.head(20)))

In [ ]:
# Query dim_feature_registry for promoted features
with engine.connect() as conn:
    try:
        reg_df = pd.read_sql(
            _text("""
                SELECT
                    feature_name,
                    lifecycle,
                    description,
                    compute_mode,
                    best_ic,
                    best_horizon,
                    promoted_at,
                    updated_at
                FROM dim_feature_registry
                ORDER BY lifecycle, feature_name
            """),
            conn,
        )
    except Exception as exc:
        print(f"Could not query dim_feature_registry: {exc}")
        reg_df = pd.DataFrame()

if reg_df.empty:
    print("No features in dim_feature_registry yet.")
    print("Promote a feature with FeaturePromoter.promote_feature() to populate this table.")
else:
    print(f"dim_feature_registry: {len(reg_df)} entries")
    display(reg_df)

<a id='part6-dashboard'></a>
## Part 6 — Interactive Dashboard

The ta_lab2 Streamlit dashboard provides an interactive interface for:

- **Pipeline Monitor** — Track data freshness, row counts, and ingestion timestamps across all tables
- **Feature Explorer** — Browse IC results, regime analysis, and feature distributions
- **Regime Viewer** — Visualize regime labels, flip frequency, and co-movement
- **Signal Lab** — Backtest signal generators with regime filtering
- **Experiment Tracker** — Compare experimental features, view BH results, trigger promotions

The cell below launches Streamlit as a **non-blocking subprocess**.  
The notebook remains interactive — you can continue running cells while the dashboard is live.

**Default URL:** http://localhost:8501

In [ ]:
import subprocess
import time

DASHBOARD_APP = str(_ROOT / "src" / "ta_lab2" / "dashboard" / "app.py")
DASHBOARD_PORT = 8501

print(f"Launching Streamlit dashboard: {DASHBOARD_APP}")
print(f"Port: {DASHBOARD_PORT}")
print()

proc = subprocess.Popen(
    [
        "streamlit", "run", DASHBOARD_APP,
        "--server.port", str(DASHBOARD_PORT),
        "--server.headless", "true",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Give Streamlit a moment to start
time.sleep(3)

if proc.poll() is None:
    print(f"Dashboard started (PID {proc.pid})")
    print(f"Open in browser: http://localhost:{DASHBOARD_PORT}")
    print()
    print("To stop the dashboard, run the next cell.")
else:
    stdout = proc.stdout.read().decode(errors="replace")
    stderr = proc.stderr.read().decode(errors="replace")
    print("Dashboard process exited early.")
    if stderr:
        print(f"stderr: {stderr[:500]}")
    if stdout:
        print(f"stdout: {stdout[:500]}")
    print()
    print("To launch manually:")
    print(f"  streamlit run {DASHBOARD_APP}")

In [ ]:
# Run this cell to stop the dashboard
try:
    if proc.poll() is None:
        proc.terminate()
        proc.wait(timeout=5)
        print(f"Dashboard stopped (PID {proc.pid}).")
    else:
        print("Dashboard process has already exited.")
except NameError:
    print("Dashboard was not started in this session.")
except Exception as exc:
    print(f"Error stopping dashboard: {exc}")

<a id='summary'></a>
## Summary and What's Next

### What We Covered

1. **Feature Registry** — Loaded `features.yaml` with `FeatureRegistry`, validated expressions,  
   and inspected spec digests that detect YAML changes between runs.

2. **Dependency DAG** — Used `resolve_experiment_dag()` to determine topological execution order,  
   ensuring features with `depends_on` constraints are computed in the right sequence.

3. **Single Experiment** — Ran `ExperimentRunner.run()` with `dry_run=True` to score a feature  
   with IC and BH-corrected p-values without writing to production tables.

4. **BH Promotion Gate** — Used `FeaturePromoter.check_bh_gate()` to evaluate whether a feature  
   passes the False Discovery Rate threshold for promotion.

5. **Batch Experiments** — Ran all experimental features in DAG order, collected results,  
   and visualized IC across features and horizons in a heatmap.

6. **Persisted Results** — Queried `cmc_feature_experiments` and `dim_feature_registry`  
   for previously stored experiment outcomes and promoted feature lifecycle state.

7. **Dashboard** — Launched the Streamlit dashboard as a non-blocking subprocess  
   for interactive monitoring and exploration.

---

### Adding a New Experimental Feature

1. Add an entry to `configs/experiments/features.yaml`:
   ```yaml
   features:
     my_new_feature:
       lifecycle: experimental
       description: "..."
       compute:
         mode: inline
         expression: "close.pct_change(3).rolling(10).mean()"
       inputs:
         - table: cmc_price_bars_multi_tf_u
           columns: [close]
       tags: [momentum]
   ```

2. Re-run this notebook (`Kernel → Restart & Run All`) to score the new feature.

3. If the BH gate passes, promote with:  
   `python -m ta_lab2.scripts.promote_feature --feature my_new_feature`

---

### Next Steps

- **Notebook 04** (coming soon) — Regime-stratified IC analysis and conditional feature evaluation
- **CLI** — `python -m ta_lab2.scripts.run_experiment` to persist results to `cmc_feature_experiments`
- **Backtest integration** — Promoted features are automatically available in the signal pipeline